In [2]:
import zipfile

# Check what files are inside the 2022 zip
with zipfile.ZipFile("/Users/aliabonanno/public_policy_project/form861/f8612022.zip") as z:
    print("\n".join(z.namelist()))

2022 EIA-861 Form.xlsx
2022 EIA-861 Instructions.pdf
Advanced_Meters_2022.xlsx
Balancing_Authority_2022.xlsx
Delivery_Companies_2022.xlsx
Demand_Response_2022.xlsx
Distribution_Systems_2022.xlsx
Dynamic_Pricing_2022.xlsx
Energy_Efficiency_2022.xlsx
Frame_2022.xlsx
Mergers_2022.xlsx
Net_Metering_2022.xlsx
Non_Net_Metering_Distributed_2022.xlsx
Operational_Data_2022.xlsx
Reliability_2022.xlsx
Sales_Ult_Cust_2022.xlsx
Sales_Ult_Cust_CS_2022.xlsx
Service_Territory_2022.xlsx
Short_Form_2022.xlsx
Utility_Data_2022.xlsx


In [4]:
import zipfile
import pandas as pd

with zipfile.ZipFile("/Users/aliabonanno/public_policy_project/form861/f8612022.zip") as z:
    
    # Check sales file
    with z.open("Sales_Ult_Cust_2022.xlsx") as f:
        sales = pd.read_excel(f, nrows=5)
    print("SALES FILE:")
    print(sales.columns.tolist())
    print(sales.head(3).to_string())
    
    print("\n" + "="*60 + "\n")
    
    # Check service territory file
    with z.open("Service_Territory_2022.xlsx") as f:
        terr = pd.read_excel(f, nrows=5)
    print("SERVICE TERRITORY FILE:")
    print(terr.columns.tolist())
    print(terr.head(3).to_string())

SALES FILE:
['Utility Characteristics', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'RESIDENTIAL', 'Unnamed: 10', 'Unnamed: 11', 'COMMERCIAL', 'Unnamed: 13', 'Unnamed: 14', 'INDUSTRIAL', 'Unnamed: 16', 'Unnamed: 17', 'TRANSPORTATION', 'Unnamed: 19', 'Unnamed: 20', 'TOTAL', 'Unnamed: 22', 'Unnamed: 23']
  Utility Characteristics      Unnamed: 1               Unnamed: 2 Unnamed: 3    Unnamed: 4                            Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8       RESIDENTIAL    Unnamed: 10 Unnamed: 11        COMMERCIAL    Unnamed: 13 Unnamed: 14        INDUSTRIAL    Unnamed: 16 Unnamed: 17    TRANSPORTATION    Unnamed: 19 Unnamed: 20             TOTAL    Unnamed: 22 Unnamed: 23
0                     NaN             NaN                      NaN        NaN           NaN                                   NaN        NaN        NaN        NaN          Revenues          Sales   Customers          Revenues          Sales

In [11]:
import pandas as pd
import zipfile
import os
import glob
import requests

# ── CONFIG ───────────────────────────────────────────────────────────────────
EIA_DIR    = "/Users/aliabonanno/public_policy_project/form861"
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

YEAR_MIN = 2012
YEAR_MAX = 2024

# Column positions (0-indexed) based on row 2 header:
# 0=Data Year, 1=Utility Number, 2=Utility Name, 3=Part, 4=Service Type,
# 5=Data Type, 6=State, 7=Ownership, 8=BA Code,
# 9=Res Revenue, 10=Res Sales MWh, 11=Res Customers
COLS_USE = [1, 2, 6, 7, 9, 10, 11]
COL_NAMES = ["utility_id", "utility_name", "state", "ownership",
             "res_revenue_000", "res_sales_mwh", "res_customers"]

# ── LOOP ─────────────────────────────────────────────────────────────────────
all_sales     = []
all_territory = []

for zip_path in sorted(glob.glob(os.path.join(EIA_DIR, "f861*.zip"))):
    basename = os.path.basename(zip_path)
    year = int(basename[4:8])
    if year < YEAR_MIN or year > YEAR_MAX:
        continue

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()

        # ── SALES ─────────────────────────────────────────────────────────
        sales_file = [f for f in names if "Sales_Ult_Cust" in f
                      and "CS" not in f and f.endswith(".xlsx")]
        if not sales_file:
            print(f"WARNING: No sales file in {basename}")
        else:
            with z.open(sales_file[0]) as f:
                # Skip rows 0,1,2 (two header rows + label row)
                # Row 2 is the real header — use it, skip rows 0 and 1
                df = pd.read_excel(f, skiprows=[0, 1], header=0)

            # Keep only the columns we need by position
            df = df.iloc[:, COLS_USE].copy()
            df.columns = COL_NAMES
            df["year"] = year

            # Convert to numeric
            for col in ["utility_id", "res_revenue_000",
                        "res_sales_mwh", "res_customers"]:
                df[col] = pd.to_numeric(df[col], errors="coerce")

            # Drop rows without a utility ID (subtotals, blanks)
            df = df.dropna(subset=["utility_id"])
            df = df[df["res_customers"] > 0]  # keep only active utilities

            all_sales.append(df)
            print(f"  {year} sales: {len(df)} utilities")

        # ── TERRITORY ─────────────────────────────────────────────────────
        terr_file = [f for f in names if "Service_Territory" in f
                     and f.endswith(".xlsx")]
        if not terr_file:
            print(f"WARNING: No territory file in {basename}")
        else:
            with z.open(terr_file[0]) as f:
                df_terr = pd.read_excel(f)

            df_terr.columns = [c.strip().lower().replace(" ", "_")
                               for c in df_terr.columns]
            df_terr = df_terr.rename(columns={
                "data_year":      "year",
                "utility_number": "utility_id",
                "state":          "state",
                "county":         "county_name"
            })
            df_terr = df_terr[["year", "utility_id",
                               "state", "county_name"]].copy()
            df_terr["utility_id"] = pd.to_numeric(
                df_terr["utility_id"], errors="coerce")
            df_terr = df_terr.dropna(subset=["utility_id", "county_name"])
            all_territory.append(df_terr)
            print(f"  {year} territory: {len(df_terr)} utility-county rows")

# ── STACK ─────────────────────────────────────────────────────────────────────
sales     = pd.concat(all_sales,     ignore_index=True)
territory = pd.concat(all_territory, ignore_index=True)

print(f"\nSales panel:     {len(sales)} rows")
print(f"Territory panel: {len(territory)} rows")
print(f"\nOwnership values:\n{sales['ownership'].value_counts().head(10)}")
print(f"\nSample sales:\n{sales.head(3).to_string()}")
print(f"\nSample territory:\n{territory.head(3).to_string()}")

# ── FIPS CROSSWALK ────────────────────────────────────────────────────────────
fips_path = os.path.join(OUTPUT_DIR, "county_fips.csv")
if not os.path.exists(fips_path):
    print("\nDownloading county FIPS crosswalk...")
    url = "https://www2.census.gov/geo/docs/reference/codes2020/national_county2020.txt"
    r = requests.get(url)
    with open(fips_path, "wb") as f:
        f.write(r.content)
    print("Downloaded.")

fips_ref = pd.read_csv(fips_path, sep="|", dtype=str)
fips_ref.columns = [c.strip().lower() for c in fips_ref.columns]
print(f"\nFIPS crosswalk columns: {fips_ref.columns.tolist()}")
print(fips_ref.head(3).to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
sales.to_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"), index=False)
territory.to_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"), index=False)
print(f"\nSaved eia861_sales.csv and eia861_territory.csv")

  2012 sales: 2530 utilities
  2015 sales: 2829 utilities
  2015 territory: 11972 utility-county rows
  2016 sales: 2901 utilities
  2016 territory: 11952 utility-county rows
  2017 sales: 2969 utilities
  2017 territory: 11827 utility-county rows
  2018 sales: 2964 utilities
  2018 territory: 11826 utility-county rows
  2019 sales: 3990 utilities
  2019 territory: 11751 utility-county rows
  2020 sales: 2375 utilities
  2020 territory: 11793 utility-county rows
  2021 sales: 2349 utilities
  2021 territory: 11686 utility-county rows
  2022 sales: 2365 utilities
  2022 territory: 11782 utility-county rows
  2023 sales: 2359 utilities
  2023 territory: 11782 utility-county rows
  2024 sales: 2328 utilities
  2024 territory: 11776 utility-county rows

Sales panel:     29959 rows
Territory panel: 118147 rows

Ownership values:
ownership
Cooperative                    8325
Municipal                      8165
Retail Power Marketer          6703
Investor Owned                 2716
Behind the

In [9]:
import pandas as pd
import zipfile

with zipfile.ZipFile("/Users/aliabonanno/public_policy_project/form861/f8612022.zip") as z:
    with z.open("Sales_Ult_Cust_2022.xlsx") as f:
        raw = pd.read_excel(f, header=None, nrows=4)

for i, row in raw.iterrows():
    print(f"Row {i}: {row.tolist()}")

Row 0: ['Utility Characteristics', nan, nan, nan, nan, nan, nan, nan, nan, 'RESIDENTIAL', nan, nan, 'COMMERCIAL', nan, nan, 'INDUSTRIAL', nan, nan, 'TRANSPORTATION', nan, nan, 'TOTAL', nan, nan]
Row 1: [nan, nan, nan, nan, nan, nan, nan, nan, nan, 'Revenues', 'Sales', 'Customers', 'Revenues', 'Sales', 'Customers', 'Revenues', 'Sales', 'Customers', 'Revenues', 'Sales', 'Customers', 'Revenues', 'Sales', 'Customers']
Row 2: ['Data Year', 'Utility Number', 'Utility Name', 'Part', 'Service Type', 'Data Type\nO = Observed\nI = Imputed', 'State', 'Ownership', 'BA Code', 'Thousand Dollars', 'Megawatthours', 'Count', 'Thousand Dollars', 'Megawatthours', 'Count', 'Thousand Dollars', 'Megawatthours', 'Count', 'Thousand Dollars', 'Megawatthours', 'Count', 'Thousand Dollars', 'Megawatthours', 'Count']
Row 3: [2022, 55, 'City of Aberdeen - (MS)', 'A', 'Bundled', 'O', 'MS', 'Municipal', 'TVA', 4345, 34383, 2588, 3684, 29241, 711, 7847, 129598, 2, 0, 0, 0, 15876, 193222, 3301]


In [15]:
import requests
import os

OUTPUT_DIR = "/Users/aliabonanno/public_policy_project/form861"

# Direct download URLs for missing years
urls = {
    "f8612013.zip": "https://www.eia.gov/electricity/data/eia861/archive/zip/f8612013.zip",
    "f8612014.zip": "https://www.eia.gov/electricity/data/eia861/archive/zip/f8612014.zip",
}

for filename, url in urls.items():
    outpath = os.path.join(OUTPUT_DIR, filename)
    print(f"Downloading {filename}...")
    r = requests.get(url, timeout=60)
    print(f"  Status: {r.status_code}, Size: {len(r.content)/1024:.0f} KB")
    with open(outpath, "wb") as f:
        f.write(r.content)
    
    # Check what's inside
    import zipfile
    with zipfile.ZipFile(outpath) as z:
        files = [f for f in z.namelist() if f.endswith(".xlsx")]
        print(f"  Contents: {files}")

  Status: 200, Size: 3170 KB
  Contents: ['Energy_Efficiency_2013.xlsx', 'Operational_Data_2013.xlsx', 'Reliability_2013.xlsx']
  Status: 200, Size: 3332 KB
  Contents: ['Energy_Efficiency_2014.xlsx', 'Operational_Data_2014.xlsx', 'Reliability_2014.xlsx']


In [33]:
import pandas as pd
import numpy as np
import math
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

# ── LOAD ─────────────────────────────────────────────────────────────────────
sales     = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"))
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
fips_ref  = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"),
                        sep="|", dtype=str)

print(f"Sales: {len(sales)} rows")
print(f"Territory: {len(territory)} rows")
print(f"FIPS ref: {len(fips_ref)} rows")

# ── HELPER: CLEAN COUNTY NAME ─────────────────────────────────────────────────
def clean_county(name):
    if pd.isna(name):
        return name
    s = str(name)
    # Strip trailing geographic suffixes
    s = pd.Series([s]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    # Standardize St. -> Saint
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ")
    # Remove periods
    s = s.replace(".", "")
    return s.strip()

# ── STEP 1: CLEAN FIPS REFERENCE ─────────────────────────────────────────────
fips_ref.columns = [c.strip().lower() for c in fips_ref.columns]
fips_ref["FIPS"] = fips_ref["statefp"].str.zfill(2) + fips_ref["countyfp"].str.zfill(3)
fips_ref["county_clean"] = fips_ref["countyname"].apply(clean_county)
fips_ref["state_abbr"]   = fips_ref["state"].str.upper()

print(f"\nSample FIPS ref after cleaning:")
print(fips_ref[["state_abbr", "county_clean", "FIPS"]].head(5).to_string())

# ── STEP 2: CLEAN TERRITORY COUNTY NAMES ─────────────────────────────────────
territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()


# ── STEP 4: DROP NON-TRADITIONAL UTILITY TYPES ───────────────────────────────
keep_ownership = ["Investor Owned", "Cooperative", "Municipal",
                  "Political Subdivision", "State", "Federal"]
sales = sales[sales["ownership"].isin(keep_ownership)].copy()

ownership_map = {
    "Investor Owned":        "IOU",
    "Cooperative":           "Cooperative",
    "Municipal":             "Municipal",
    "Political Subdivision": "Municipal",
    "State":                 "Municipal",
    "Federal":               "Municipal",
}
sales["ownership_type"] = sales["ownership"].map(ownership_map)
print(f"\nOwnership after cleaning:\n{sales['ownership_type'].value_counts()}")

# ── STEP 5: MERGE TERRITORY TO FIPS ──────────────────────────────────────────
territory_fips = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"],
    how="left"
)

n_matched   = territory_fips["FIPS"].notna().sum()
n_unmatched = territory_fips["FIPS"].isna().sum()
print(f"\nTerritory FIPS match: {n_matched} matched, {n_unmatched} unmatched "
      f"({n_unmatched/len(territory_fips):.1%} unmatched)")

unmatched = (territory_fips[territory_fips["FIPS"].isna()]
             [["state_abbr", "county_name", "county_clean"]]
             .drop_duplicates().head(10))
print(f"\nSample unmatched:\n{unmatched.to_string()}")

# ── STEP 6: MERGE SALES TO TERRITORY ─────────────────────────────────────────
utility_county = territory_fips.merge(
    sales[["utility_id", "year", "state", "ownership_type",
           "res_revenue_000", "res_sales_mwh", "res_customers"]],
    on=["utility_id", "year"],
    how="inner"
)

print(f"\nUtility-county panel: {len(utility_county)} rows")
print(f"Unique FIPS: {utility_county['FIPS'].nunique()}")
print(f"Years: {sorted(utility_county['year'].unique())}")

# ── STEP 7: AGGREGATE TO COUNTY LEVEL ────────────────────────────────────────
county_agg = (utility_county.dropna(subset=["FIPS"])
              .groupby(["FIPS", "year"])
              .agg(
                  res_revenue_000_total = ("res_revenue_000", "sum"),
                  res_sales_mwh_total   = ("res_sales_mwh",   "sum"),
                  res_customers_total   = ("res_customers",   "sum"),
                  n_utilities           = ("utility_id",      "nunique"),
              )
              .reset_index())

# Implied price — NaN if revenue missing (e.g. 2019)
county_agg["implied_price_per_mwh"] = np.where(
    (county_agg["res_revenue_000_total"] > 0) &
    (county_agg["res_sales_mwh_total"]   > 0),
    (county_agg["res_revenue_000_total"] * 1000) / county_agg["res_sales_mwh_total"],
    np.nan
)

# Sales per customer
county_agg["res_sales_per_customer"] = np.where(
    county_agg["res_customers_total"] > 0,
    county_agg["res_sales_mwh_total"] / county_agg["res_customers_total"],
    np.nan
)

# Log transforms
county_agg["log_res_sales_per_customer"] = np.log(
    county_agg["res_sales_per_customer"].replace(0, np.nan))
county_agg["log_implied_price"] = np.log(
    county_agg["implied_price_per_mwh"].replace(0, np.nan))

# Dominant ownership type (largest utility by customer count)
ownership_by_county = (utility_county
                       .dropna(subset=["FIPS", "res_customers"])
                       .sort_values("res_customers", ascending=False)
                       .groupby(["FIPS", "year"])
                       .agg(dominant_ownership_type=("ownership_type", "first"))
                       .reset_index())

county_agg = county_agg.merge(ownership_by_county, on=["FIPS", "year"], how="left")

print(f"\nCounty-year panel: {len(county_agg)} rows")
print(f"Unique counties: {county_agg['FIPS'].nunique()}")
print(f"Years: {sorted(county_agg['year'].unique())}")

print(f"\nSummary stats:")
print(county_agg[["res_sales_per_customer", "implied_price_per_mwh",
                   "res_customers_total"]].describe().round(2))

# Sanity check: Autauga County
print(f"\nAutauga County (01001):")
print(county_agg[county_agg["FIPS"] == "01001"][
    ["FIPS", "year", "res_sales_per_customer",
     "implied_price_per_mwh", "dominant_ownership_type"]
].to_string())

county_agg.to_csv(os.path.join(OUTPUT_DIR, "eia861_county_panel.csv"), index=False)
print(f"\nSaved eia861_county_panel.csv")

Sales: 29945 rows
Territory: 118147 rows
FIPS ref: 3235 rows

Sample FIPS ref after cleaning:
  state_abbr county_clean   FIPS
0         AL      autauga  01001
1         AL      baldwin  01003
2         AL      barbour  01005
3         AL         bibb  01007
4         AL       blount  01009

Ownership after cleaning:
ownership_type
Municipal      9023
Cooperative    8325
IOU            2716
Name: count, dtype: int64

Territory FIPS match: 117690 matched, 637 unmatched (0.5% unmatched)

Sample unmatched:
     state_abbr                county_name               county_clean
119          AK  Prince of Wales Ketchikan  prince of wales ketchikan
120          AK      Skagway Hoonah Angoon      skagway hoonah angoon
121          AK              Yukon Koyukuk              yukon koyukuk
124          AK              Kodiak Island                     kodiak
127          AK               Wade Hampton               wade hampton
474          MD             Prince Georges             prince georges
6

In [19]:
import pandas as pd

sales = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_sales.csv")

# Check 2019 specifically for Autauga-area utilities
print(sales[sales["year"] == 2019].head(5).to_string())
print("\nColumn order check — 2019 vs 2020:")
print(sales[sales["year"] == 2019][["utility_id","res_revenue_000","res_sales_mwh","res_customers"]].head(3).to_string())
print(sales[sales["year"] == 2020][["utility_id","res_revenue_000","res_sales_mwh","res_customers"]].head(3).to_string())

       utility_id              utility_name state    ownership  res_revenue_000  res_sales_mwh  res_customers  year
14193        34.0  City of Abbeville - (SC)    SC    Municipal              NaN         4563.0        33000.0  2019
14194        55.0   City of Aberdeen - (MS)    MS    Municipal              NaN         3930.0        34781.0  2019
14195        59.0  City of Abbeville - (LA)    LA    Municipal              NaN         5907.5        59232.0  2019
14196        84.0       A & N Electric Coop    MD  Cooperative              NaN          273.5         2094.0  2019
14197        84.0       A & N Electric Coop    VA  Cooperative              NaN        44102.0       357988.0  2019

Column order check — 2019 vs 2020:
       utility_id  res_revenue_000  res_sales_mwh  res_customers
14193        34.0              NaN         4563.0        33000.0
14194        55.0              NaN         3930.0        34781.0
14195        59.0              NaN         5907.5        59232.0
       u

In [27]:
import pandas as pd
import zipfile
import os
import glob
import requests

# ── CONFIG ───────────────────────────────────────────────────────────────────
EIA_DIR    = "/Users/aliabonanno/public_policy_project/form861"
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

YEAR_MIN = 2012
YEAR_MAX = 2024

# Column positions (0-indexed) for residential revenue, sales, customers
# 2019 has an extra "Short Form" column at position 9 shifting everything right
COLS_BY_YEAR = {
    2019:      [1, 2, 6, 7, 10, 11, 12],
    "default": [1, 2, 6, 7,  9, 10, 11],
}
COL_NAMES = ["utility_id", "utility_name", "state", "ownership",
             "res_revenue_000", "res_sales_mwh", "res_customers"]

# ── LOOP ─────────────────────────────────────────────────────────────────────
all_sales     = []
all_territory = []

for zip_path in sorted(glob.glob(os.path.join(EIA_DIR, "f861*.zip"))):
    basename = os.path.basename(zip_path)
    year = int(basename[4:8])
    if year < YEAR_MIN or year > YEAR_MAX:
        continue

    cols_use = COLS_BY_YEAR.get(year, COLS_BY_YEAR["default"])

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()

        # ── SALES ─────────────────────────────────────────────────────────
        sales_file = [f for f in names if "Sales_Ult_Cust" in f
                      and "CS" not in f and f.endswith(".xlsx")]
        if not sales_file:
            print(f"WARNING: No sales file in {basename}")
        else:
            with z.open(sales_file[0]) as f:
                df = pd.read_excel(f, skiprows=[0, 1], header=0)

            df = df.iloc[:, cols_use].copy()
            df.columns = COL_NAMES
            df["year"] = year

            for col in ["utility_id", "res_revenue_000",
                        "res_sales_mwh", "res_customers"]:
                df[col] = pd.to_numeric(df[col], errors="coerce")

            df = df.dropna(subset=["utility_id"])
            df = df[df["res_customers"] > 0]

            all_sales.append(df)
            print(f"  {year} sales: {len(df)} utilities "
                  f"(cols: {cols_use})")

        # ── TERRITORY ─────────────────────────────────────────────────────
        terr_file = [f for f in names if "Service_Territory" in f
                     and f.endswith(".xlsx")]
        if not terr_file:
            print(f"WARNING: No territory file in {basename}")
        else:
            with z.open(terr_file[0]) as f:
                df_terr = pd.read_excel(f)

            df_terr.columns = [c.strip().lower().replace(" ", "_")
                               for c in df_terr.columns]
            df_terr = df_terr.rename(columns={
                "data_year":      "year",
                "utility_number": "utility_id",
                "state":          "state",
                "county":         "county_name"
            })
            df_terr = df_terr[["year", "utility_id",
                               "state", "county_name"]].copy()
            df_terr["utility_id"] = pd.to_numeric(
                df_terr["utility_id"], errors="coerce")
            df_terr = df_terr.dropna(subset=["utility_id", "county_name"])
            all_territory.append(df_terr)
            print(f"  {year} territory: {len(df_terr)} utility-county rows")

# ── STACK ─────────────────────────────────────────────────────────────────────
sales     = pd.concat(all_sales,     ignore_index=True)
territory = pd.concat(all_territory, ignore_index=True)

print(f"\nSales panel:     {len(sales)} rows")
print(f"Territory panel: {len(territory)} rows")
print(f"\nOwnership values:\n{sales['ownership'].value_counts().head(8)}")

# Sanity check 2019 vs 2020 for same utility
print(f"\n2019 vs 2020 sanity check (utility 55):")
print(sales[sales["utility_id"] == 55][
    ["utility_id", "year", "res_revenue_000",
     "res_sales_mwh", "res_customers"]
].to_string())

# ── FIPS CROSSWALK ────────────────────────────────────────────────────────────
fips_path = os.path.join(OUTPUT_DIR, "county_fips.csv")
if not os.path.exists(fips_path):
    print("\nDownloading county FIPS crosswalk...")
    url = "https://www2.census.gov/geo/docs/reference/codes2020/national_county2020.txt"
    r = requests.get(url)
    with open(fips_path, "wb") as f:
        f.write(r.content)
    print("Downloaded.")

# ── SAVE ──────────────────────────────────────────────────────────────────────
sales.to_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"), index=False)
territory.to_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"), index=False)
print(f"\nSaved eia861_sales.csv and eia861_territory.csv")

  2012 sales: 2530 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2015 sales: 2829 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2015 territory: 11972 utility-county rows
  2016 sales: 2901 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2016 territory: 11952 utility-county rows
  2017 sales: 2969 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2017 territory: 11827 utility-county rows
  2018 sales: 2964 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2018 territory: 11826 utility-county rows
  2019 sales: 3976 utilities (cols: [1, 2, 6, 7, 10, 11, 12])
  2019 territory: 11751 utility-county rows
  2020 sales: 2375 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2020 territory: 11793 utility-county rows
  2021 sales: 2349 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2021 territory: 11686 utility-county rows
  2022 sales: 2365 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2022 territory: 11782 utility-county rows
  2023 sales: 2359 utilities (cols: [1, 2, 6, 7, 9, 10, 11])
  2023 territory: 11782 utility-count

In [6]:
import requests
import os

OUTPUT_DIR = "/Users/aliabonanno/public_policy_project/form861m"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE = "https://www.eia.gov/electricity/data/eia861m"

FILES = {
    2015: f"{BASE}/archive/xls/f8262015.xls",
    2016: f"{BASE}/archive/xls/f8262016.xls",
    2017: f"{BASE}/archive/xls/retail_sales_2017.xlsx",
    2018: f"{BASE}/archive/xls/retail_sales_2018.xlsx",
    2019: f"{BASE}/archive/xls/retail_sales_2019.xlsx",
    2020: f"{BASE}/archive/xls/retail_sales_2020.xlsx",
    2021: f"{BASE}/archive/xls/sales_ult_cust_2021.xlsx",
    2022: f"{BASE}/archive/xls/sales_ult_cust_2022.xlsx",
    2023: f"{BASE}/archive/xls/sales_ult_cust_2023.xlsx",
    2024: f"{BASE}/archive/xls/sales_ult_cust_2024.xlsx",
}

for year, url in FILES.items():
    ext = ".xls" if url.endswith(".xls") else ".xlsx"
    filename = f"eia861m_{year}{ext}"
    outpath = os.path.join(OUTPUT_DIR, filename)
    print(f"Downloading {year}...")
    r = requests.get(url, timeout=60)
    print(f"  Status: {r.status_code}, Size: {len(r.content)/1024:.0f} KB")
    if r.status_code == 200:
        with open(outpath, "wb") as f:
            f.write(r.content)
    else:
        print(f"  FAILED: {url}")

print("\nDone.")

  Status: 200, Size: 2198 KB
  Status: 200, Size: 1332 KB
  Status: 200, Size: 1364 KB
  Status: 200, Size: 1404 KB
  Status: 200, Size: 1409 KB
  Status: 200, Size: 1403 KB
  Status: 200, Size: 1373 KB
  Status: 200, Size: 1384 KB
  Status: 200, Size: 1408 KB
  Status: 200, Size: 1459 KB

Done.


In [8]:
import pandas as pd

df = pd.read_excel("/Users/aliabonanno/public_policy_project/form861m/eia861m_2024.xlsx", nrows=5)
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head(5).to_string())

Shape: (5, 22)

First 5 rows:
  UTILITY CHARATERISTICS Unnamed: 1      Unnamed: 2                         Unnamed: 3 Unnamed: 4      Unnamed: 5   Unnamed: 6        RESIDENTIAL     Unnamed: 8 Unnamed: 9         COMMERCIAL    Unnamed: 11 Unnamed: 12         INDUSTRIAL    Unnamed: 14 Unnamed: 15     TRANSPORTATION    Unnamed: 17 Unnamed: 18              TOTAL    Unnamed: 20 Unnamed: 21
0                    NaN        NaN             NaN                                NaN        NaN             NaN          NaN            Revenue          Sales  Customers            Revenue          Sales   Customers            Revenue          Sales   Customers            Revenue          Sales   Customers            Revenue          Sales   Customers
1                   Year      Month  Utility Number                       Utility Name      State       Ownership  Data Status  Thousands Dollars  Megawatthours      Count  Thousands Dollars  Megawatthours       Count  Thousands Dollars  Megawatthours       

In [10]:
import pandas as pd
import numpy as np
import glob
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
EIA_M_DIR  = "/Users/aliabonanno/public_policy_project/form861m"
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

YEAR_MIN = 2015
YEAR_MAX = 2024

# Column positions (0-indexed) based on row 1 header:
# 0=Year, 1=Month, 2=Utility Number, 3=Utility Name, 4=State, 5=Ownership
# 6=Data Status, 7=Res Revenue, 8=Res Sales MWh, 9=Res Customers
COLS_USE  = [0, 1, 2, 3, 4, 5, 7, 8, 9]
COL_NAMES = ["year", "month", "utility_id", "utility_name", "state",
             "ownership", "res_revenue_000", "res_sales_mwh", "res_customers"]

# ── LOOP ─────────────────────────────────────────────────────────────────────
all_years = []

for fpath in sorted(glob.glob(os.path.join(EIA_M_DIR, "eia861m_*.xl*"))):
    basename = os.path.basename(fpath)
    year     = int(basename.replace("eia861m_", "").split(".")[0])

    if year < YEAR_MIN or year > YEAR_MAX:
        continue

    df = pd.read_excel(fpath, skiprows=[0, 1], header=0)
    df = df.iloc[:, COLS_USE].copy()
    df.columns = COL_NAMES

    # Convert to numeric
    for col in ["utility_id", "year", "month", "res_revenue_000",
                "res_sales_mwh", "res_customers"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop state adjustment rows (utility_id == 0) and missing utility IDs
    df = df[df["utility_id"] > 0]
    df = df.dropna(subset=["utility_id", "year", "month"])

    # Drop non-traditional ownership types
    keep_ownership = ["Investor Owned", "Cooperative", "Municipal",
                      "Political Subdivision", "State", "Federal"]
    df = df[df["ownership"].isin(keep_ownership)].copy()

    # Simplify ownership
    ownership_map = {
        "Investor Owned":        "IOU",
        "Cooperative":           "Cooperative",
        "Municipal":             "Municipal",
        "Political Subdivision": "Municipal",
        "State":                 "Municipal",
        "Federal":               "Municipal",
    }
    df["ownership_type"] = df["ownership"].map(ownership_map)

    # Keep only rows with positive customers
    df = df[df["res_customers"] > 0]

    all_years.append(df)
    print(f"  {year}: {len(df)} utility-month rows, "
          f"{df['month'].nunique()} months, "
          f"{df['utility_id'].nunique()} utilities")

# ── STACK ─────────────────────────────────────────────────────────────────────
sales_m = pd.concat(all_years, ignore_index=True)
sales_m = sales_m.sort_values(["utility_id", "year", "month"]).reset_index(drop=True)

print(f"\nMonthly sales panel: {len(sales_m)} rows")
print(f"Years: {sorted(sales_m['year'].unique())}")
print(f"Unique utilities: {sales_m['utility_id'].nunique()}")
print(f"\nOwnership distribution:\n{sales_m['ownership_type'].value_counts()}")
print(f"\nSample rows:")
print(sales_m.head(5).to_string())

# Sanity check: utility 55 (City of Aberdeen MS) — should have 12 months per year
util55 = sales_m[sales_m["utility_id"] == 55][
    ["utility_id", "year", "month", "res_revenue_000",
     "res_sales_mwh", "res_customers"]
]
print(f"\nUtility 55 (City of Aberdeen MS) — 2023:")
print(util55[util55["year"] == 2023].to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
sales_m.to_csv(os.path.join(OUTPUT_DIR, "eia861m_sales.csv"), index=False)
print(f"\nSaved eia861m_sales.csv")

  2015: 4859 utility-month rows, 12 months, 334 utilities
  2016: 4859 utility-month rows, 12 months, 334 utilities
  2017: 4860 utility-month rows, 12 months, 336 utilities
  2018: 4849 utility-month rows, 12 months, 335 utilities
  2019: 4851 utility-month rows, 12 months, 335 utilities
  2020: 4858 utility-month rows, 12 months, 336 utilities
  2021: 4848 utility-month rows, 12 months, 335 utilities
  2022: 4835 utility-month rows, 12 months, 334 utilities
  2023: 4836 utility-month rows, 12 months, 334 utilities
  2024: 4800 utility-month rows, 12 months, 331 utilities

Monthly sales panel: 48455 rows
Years: [2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, 2024.0]
Unique utilities: 339

Ownership distribution:
ownership_type
IOU            22419
Cooperative    14319
Municipal      11717
Name: count, dtype: int64

Sample rows:
     year  month  utility_id         utility_name state    ownership  res_revenue_000  res_sales_mwh  res_customers ownership_type
0  

In [12]:
import pandas as pd

sales_m   = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861m_sales.csv")
territory = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_territory.csv")

# Quick merge to see county coverage
merged = sales_m.merge(territory[["utility_id", "state", "county_name"]],
                       on=["utility_id", "state"], how="left")

print(f"Monthly utility-months: {len(sales_m)}")
print(f"After territory merge: {len(merged)}")
print(f"County names matched: {merged['county_name'].notna().sum()}")
print(f"Unique counties: {merged['county_name'].nunique()}")

Monthly utility-months: 48455
After territory merge: 5088509
County names matched: 5088509
Unique counties: 1679


In [14]:
import pandas as pd

sales_m   = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861m_sales.csv")
territory = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_territory.csv")

# Merge on utility_id + year + state to prevent fan-out
sales_m["year"]   = sales_m["year"].astype(int)
territory["year"] = territory["year"].astype(int)

merged = sales_m.merge(territory[["utility_id", "year", "state", "county_name"]],
                       on=["utility_id", "year", "state"], how="left")

print(f"Monthly utility-month-county rows: {len(merged)}")
print(f"County names matched: {merged['county_name'].notna().sum()}")
print(f"Unique counties: {merged['county_name'].nunique()}")
print(f"Years: {sorted(merged['year'].unique())}")
print(f"\nSample:")
print(merged.head(5)[["year","month","utility_id","state","county_name","res_sales_mwh"]].to_string())

Monthly utility-month-county rows: 511447
County names matched: 511423
Unique counties: 1679
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Sample:
   year  month  utility_id state  county_name  res_sales_mwh
0  2015    1.0        84.0    MD     Somerset        243.862
1  2015    1.0        84.0    VA     Accomack      37917.786
2  2015    1.0        84.0    VA  Northampton      37917.786
3  2015    2.0        84.0    MD     Somerset        292.881
4  2015    2.0        84.0    VA     Accomack      42758.573
